# 🧠 دفتر النموذج — `NIG-TimeNet v2` (نسخة مُنظَّفة)

هذا الدفتر يُعرِّف **معمارية النموذج فقط**. كل ما لا علاقة له بتعريف
النموذج (تحميل دفاتر أخرى من Google Drive، تجهيز بيانات، تدريب، حفظ/تحميل
أوزان، استدعاءات تحليل) أُزيل من هنا وانتقل إلى دفتر `main` الموحّد، الذي
يستورد `build_model_fn` من هذا الدفتر ويستخدمه مع بيانات حقيقية.

## سجلّ التنظيف (لماذا أُزيل كل شيء آخر)

النسخة السابقة من هذا الدفتر (24 خلية) كانت تراكماً من تجارب متتالية:
ثلاث محاولات تحميل دفتر آخر من Drive بروابط `file_id` مختلفة، خلية كود
عربي مُترجَم آلياً (`استيراد`، `طباعة`، `دالة`...) غير قابلة للتنفيذ إطلاقاً،
معماريتان قديمتان (`build_advanced_causal_transformer`،
`build_ultimate_hypertimenet`) بها كود مكسور بنفس الطريقة ومُستبدَلتان فعلياً
بـ`build_nig_timenet_v2` (توثّق ذلك بنفسها في docstring الأصلي: "Drop-in
replacement")، مسارات Colab محلّية مُثبَّتة (`/content/saved_data`)، وخلايا
تدريب/تقييم تجريبية. كل ذلك أُزيل — لم يبقَ سوى المعمارية الفعلية المُستخدَمة
(`build_nig_timenet_v2`) وإعداداتها (`MODEL_CONFIG`).

## رؤوس مُوصَّفة (pluggable heads)

المعمارية القديمة كانت تبني رأس NIG واحداً فقط، مُثبَّتاً بالكود، لكل من
high/low/close. خط الأنابيب (`crypto_data_pipeline_v6`) يُفعِّل افتراضياً
**رأسي تصنيف وانحدار معاً** لكل هدف (`enabled_heads`) — فجوة حقيقية كانت
قائمة بين ما يُعِدّه خط الأنابيب وما ينتجه النموذج.

بدل تثبيت رأس تصنيف بالكود، صُمِّم **سجلّ رؤوس** (`HEAD_REGISTRY`) يحصر
قرار "ما هي المخرجات وما أسماؤها" في دوال صغيرة منفصلة:

* `register_head_type(name)` — يُسجِّل دالة بناء رأس جديد بسطر واحد.
* `MODEL_CONFIG['head_types']` — قاموس `{هدف: [أنواع الرؤوس]}` يتحكّم فعلياً
  في أي رأس يُبنى لأي هدف، بلا لمس كود المعمارية نفسه.

## تحديث: رؤوس التصنيف مُفعَّلة الآن افتراضياً

`MODEL_CONFIG['head_types']` أصبح يتضمّن `binary_classification` لكل من
high/low/close، جنباً إلى جنب مع `nig_regression` — مطابقةً لـ`enabled_heads`
الافتراضي في خط الأنابيب (كل هدف له رأسا `_class`/`_reg` معاً). النموذج
الافتراضي الآن يُخرج 24 مفتاحاً (21 من NIG + 3 احتمالات تصنيف ثنائي)
بدل 21 فقط في النسخة السابقة.

تعطيلها أو تعديلها يبقى سطراً واحداً بلا لمس كود المعمارية:

```python
# تعطيل التصنيف تماماً، انحدار NIG فقط (سلوك النسخة السابقة):
MODEL_CONFIG['head_types'] = {t: ['nig_regression'] for t in ('high', 'low', 'close')}

# إضافة رأس تصنيف متعدّد الفئات (مثلاً نظام سوق) فوق ما هو مُفعَّل أصلاً:
MODEL_CONFIG['head_types']['close'].append('multiclass_classification')
```

راجع الاختبار الذاتي في آخر الدفتر — يُغطّي كلا الاتجاهين (تفعيل/تعطيل) فعلياً.

## تحديث PR #7: مقاومة الحفظ (`ANTI_MEMORIZATION_CONFIG`)

النموذج السليم ليس نموذجاً «لا يستطيع» الحفظ — أي شبكة بهذه السعة تحفظ تسميات عشوائية إن دُرِّبت كفاية —
بل نموذج يكون فيه **النمط البسيط أرخص من الحفظ**، فيفشل بصدق حين لا توجد معلومة بدل أن يحفظ train.
خيارات جديدة في `build_nig_timenet_v2` (كلها افتراضياً = السلوك القديم تماماً، فنموذج مُدرَّب سابقاً
يُستأنف بلا تغيير):

| الخيار | ما يفعله | لماذا |
|---|---|---|
| `input_noise_std` | ضجيج غاوسي على z أثناء التدريب | لا يرى النموذج نفس القيم الدقيقة مرّتين فلا يحفظها |
| `feature_dropout` | إسقاط قناة ميزة كاملة لكل عيّنة (`SpatialDropout1D`) | يمنع الاعتماد على ميزة واحدة تصلح معرِّفاً |
| `stats_mode` | `full` / `symlog` / `none` لمسار `[mean, log std]` | الطريق الوحيد الذي يصل منه **مستوى** ميزة غير مُطبَّعة (سعر خام…) = معرِّف عملة/فترة |
| `input_clip` | قصّ ناعم `c·tanh(z/c)` | ميزة شبه ثابتة تقفز مرّة تُعطي قيمة متطرّفة في خطوة واحدة |
| `linear_path_l2` | L2 للمسار الخطّي (كان 1e-4 مُثبَّتاً) | `Flatten(z)` مُدخل بحجم seq_len×n_features |
| `level_passthrough` | يمرّر المدخل الخام بعد `SymLog` إلى المحوّل بجانب z | InstanceNorm يمحو مستوى الميزة داخل النافذة، فالفرق بين عملة متقلّبة وهادئة (إشارة H003) لا يصل للمحوّل إلا عبر stats |

`ANTI_MEMORIZATION_CONFIG` (القسم ٩) يجمع القيم المُختبَرة فعلياً؛ الأرقام والتجارب المرفوضة في
`docs/research/anti_memorization_pr7.md`.


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers, initializers

register = tf.keras.utils.register_keras_serializable(package="nigts")

## 1) تطبيع المدخل (RevIN-style, نصف المدخل فقط)

In [ ]:
@register
class InstanceNorm(layers.Layer):
    """تقييس لكل نافذة ولكل ميزة على حدة. تُرجع (z, stats) حيث stats =
    [mean, log std] لكل ميزة — بحيث لا يُفقَد مستوى التقلّب.
    بلا نصف إعادة تسوية: الأهداف أصلاً بلا وحدة قياس (نسبية للسعر الحالي).

    حارسان ضد الحفظ حين يكون تطبيع البيانات سيئاً (كلاهما معطَّل افتراضياً = السلوك القديم تماماً):

    * clip: قصّ ناعم z → clip·tanh(z/clip). z داخل النافذة محدود أصلاً بـ√(T−1)، لكن ميزة شبه ثابتة
      داخل النافذة تقفز مرّة واحدة تُعطي ±√(T−1) في خطوة واحدة (بصمة فريدة لتلك العيّنة).
    * stats_mode: مسار الإحصاءات هو **الطريق الوحيد** الذي يصل منه المستوى المطلق للميزة إلى النموذج
      (z يُلغيه). ميزة غير مُطبَّعة (سعر خام، حجم بالدولار، عمر العملة) تجعل mean فيه رقماً فريداً لكل
      عملة/فترة — معرِّفاً يحفظ به النموذج «أيّ عملة في أيّ يوم» بدل نمط قابل للتعميم (قيس فعلياً،
      docs/research/anti_memorization_pr7.md).
        "full"   : [mean, log std] كما هي (القديم).
        "symlog" : mean → sign·log1p|mean| و log std مقصوص في [-stats_clip, stats_clip]: يبقى مستوى
                   التقلّب ويُضغط أيّ مستوى خام لعدّة وحدات فقط بدل 10^5.
        "none"   : بلا مسار إحصاءات إطلاقاً (stats=None)."""

    STATS_MODES = ("full", "symlog", "none")

    def __init__(self, eps=1e-4, affine=True, clip=None, stats_mode="full", stats_clip=8.0, **kw):
        super().__init__(**kw)
        if stats_mode not in self.STATS_MODES:
            raise ValueError(f"stats_mode يجب أن يكون واحداً من {self.STATS_MODES}، لا {stats_mode!r}")
        self.eps, self.affine = eps, affine
        self.clip, self.stats_mode, self.stats_clip = clip, stats_mode, stats_clip

    def build(self, input_shape):
        f = int(input_shape[-1])
        if self.affine:
            self.gamma = self.add_weight(name="gamma", shape=(f,), initializer="ones")
            self.beta = self.add_weight(name="beta", shape=(f,), initializer="zeros")
        super().build(input_shape)

    def call(self, x):
        mean = tf.reduce_mean(x, axis=1, keepdims=True)
        var = tf.math.reduce_variance(x, axis=1, keepdims=True)
        std = tf.sqrt(var + self.eps)
        z = (x - mean) / std
        if self.clip:
            z = self.clip * tf.tanh(z / self.clip)
        if self.affine:
            z = z * self.gamma + self.beta
        m, log_s = tf.squeeze(mean, 1), tf.math.log(tf.squeeze(std, 1))
        if self.stats_mode == "symlog":
            m = tf.sign(m) * tf.math.log1p(tf.abs(m))
            log_s = tf.clip_by_value(log_s, -self.stats_clip, self.stats_clip)
        stats = tf.concat([m, log_s], axis=-1)
        return z, stats

    def compute_output_shape(self, input_shape):
        b, t, f = input_shape
        return (b, t, f), (b, 2 * f)

    def get_config(self):
        return {**super().get_config(), "eps": self.eps, "affine": self.affine, "clip": self.clip,
                "stats_mode": self.stats_mode, "stats_clip": self.stats_clip}


@register
class SymLog(layers.Layer):
    """sign(x)·log1p(|x|) ثم قصّ في [-clip, clip]: تمرير **المستوى** الخام للميزة (لا z) بمقياس محصور.
    ≈ الهوية للقيم الصغيرة (|x|<1، أي ميزات خط الأنابيب المُطبَّعة أصلاً)، ويضغط أي مستوى غير مُطبَّع
    (سعر خام 6.5e4 → 11) بدل أن يُغرق الطبقة التالية."""

    def __init__(self, clip=8.0, **kw):
        super().__init__(**kw)
        self.clip = clip

    def call(self, x):
        y = tf.sign(x) * tf.math.log1p(tf.abs(x))
        return tf.clip_by_value(y, -self.clip, self.clip) if self.clip else y

    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        return {**super().get_config(), "clip": self.clip}

## 2) تفكيك مقاييس زمنية سببي (يمنع تسرّب المستقبل + بقايا وهمية)

In [ ]:
@register
class CausalMultiScaleDecomp(layers.Layer):
    """trend = مزيج محدَّب مُتعلَّم من متوسطات متحرّكة سابقة (حشو بتكرار الطرف
    الأيسر). seasonal = x - trend -> trend + seasonal == x تماماً (بلا مسار
    بقايا وهمي). النوافذ اللاحقة تستخدم الماضي فقط، وموثوقة بنفس القدر عند
    آخر خطوة زمنية — وهي الخطوة التي يقرؤها الرأس."""

    def __init__(self, kernel_sizes=(3, 5, 9, 17), **kw):
        super().__init__(**kw)
        self.kernel_sizes = tuple(kernel_sizes)
        self.pools = [layers.AveragePooling1D(pool_size=k, strides=1, padding="valid")
                      for k in self.kernel_sizes]
        self.gate = layers.Dense(len(self.kernel_sizes), name="scale_gate")

    def build(self, input_shape):
        # يبني الأبناء صراحةً (لا انتظار أول call) — إلزامي لحفظ/تحميل
        # النموذج كاملاً (model.save/load_model) بلا إعادة بناء عبر
        # build_model_fn: بلا هذا، Keras لا يعرف شكل أوزان self.gate وقت
        # استعادتها من ملف الحفظ فيرفض التحميل بخطأ "never built".
        f = int(input_shape[-1])
        self.gate.build(tuple(input_shape[:-1]) + (2 * f,))
        super().build(input_shape)

    def call(self, x):
        trends = []
        for k, pool in zip(self.kernel_sizes, self.pools):
            pad = tf.repeat(x[:, :1, :], repeats=k - 1, axis=1)
            trends.append(pool(tf.concat([pad, x], axis=1)))
        trends = tf.stack(trends, axis=1)
        rough = tf.reduce_mean(tf.abs(x[:, 1:, :] - x[:, :-1, :]), axis=1)
        g = tf.concat([x[:, -1, :], rough], axis=-1)
        w = tf.nn.softmax(self.gate(g), axis=-1)
        trend = tf.einsum("bktf,bk->btf", trends, w)
        return trend, x - trend

    def compute_output_shape(self, input_shape):
        return input_shape, input_shape

    def get_config(self):
        return {**super().get_config(), "kernel_sizes": list(self.kernel_sizes)}

## 3) تضمين رُقَع (patches) + مواضع مطلقة مُتعلَّمة

In [ ]:
def num_patches(seq_len, patch_len, stride):
    pad = (stride - (seq_len - patch_len) % stride) % stride
    return (seq_len + pad - patch_len) // stride + 1, pad


@register
class PatchEmbedding(layers.Layer):
    """رُقَع مُتراكبة (خلط قنوات خطّي) مُحاذاة بحيث تنتهي آخر رقعة عند آخر
    خطوة زمنية تماماً. patch_len=1, stride=1 يُعيد نفس سلوك رمز لكل خطوة."""

    def __init__(self, d_model, patch_len=4, stride=2, **kw):
        super().__init__(**kw)
        self.d_model, self.patch_len, self.stride = d_model, patch_len, stride
        self.proj = layers.Conv1D(d_model, patch_len, strides=stride, padding="valid", name="patch_proj")

    def build(self, input_shape):
        t = int(input_shape[1])
        assert t >= self.patch_len, "seq_len must be >= patch_len"
        self.n_tokens, self.pad = num_patches(t, self.patch_len, self.stride)
        self.pos = self.add_weight(name="pos_emb", shape=(1, self.n_tokens, self.d_model),
                                    initializer=initializers.RandomNormal(stddev=0.02))
        self.proj.build(input_shape)  # لنفس سبب CausalMultiScaleDecomp.build أعلاه
        super().build(input_shape)

    def call(self, x):
        if self.pad > 0:
            x = tf.concat([tf.repeat(x[:, :1, :], self.pad, axis=1), x], axis=1)
        return self.proj(x) + self.pos

    def get_config(self):
        return {**super().get_config(), "d_model": self.d_model,
                "patch_len": self.patch_len, "stride": self.stride}

## 4) كتلة محوّل (RMSNorm ما قبل الطبقة، GQA + انحياز نسبي، SwiGLU)

In [ ]:
@register
class RMSNorm(layers.Layer):
    def __init__(self, epsilon=1e-6, **kw):
        super().__init__(**kw)
        self.epsilon = epsilon

    def build(self, input_shape):
        self.scale = self.add_weight(name="scale", shape=(int(input_shape[-1]),), initializer="ones")
        super().build(input_shape)

    def call(self, x):
        ms = tf.reduce_mean(tf.square(x), axis=-1, keepdims=True)
        return self.scale * x * tf.math.rsqrt(ms + self.epsilon)

    def get_config(self):
        return {**super().get_config(), "epsilon": self.epsilon}


@register
class RelativeGQAttention(layers.Layer):
    def __init__(self, d_model, num_heads, num_kv_heads=None, max_rel_pos=16,
                 causal=False, window=None, dropout=0.0, out_std=0.02, **kw):
        super().__init__(**kw)
        num_kv_heads = num_kv_heads or num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        assert num_heads % num_kv_heads == 0, "num_heads must be divisible by num_kv_heads"
        self.d_model, self.h, self.hkv = d_model, num_heads, num_kv_heads
        self.dh = d_model // num_heads
        self.max_rel_pos, self.causal, self.window = max_rel_pos, causal, window
        self.dropout_rate, self.out_std = dropout, out_std
        self.wq = layers.Dense(self.h * self.dh, use_bias=False)
        self.wk = layers.Dense(self.hkv * self.dh, use_bias=False)
        self.wv = layers.Dense(self.hkv * self.dh, use_bias=False)
        self.wo = layers.Dense(d_model, use_bias=False,
                                kernel_initializer=initializers.TruncatedNormal(stddev=out_std))
        self.drop = layers.Dropout(dropout)

    def build(self, input_shape):
        self.rel_bias = self.add_weight(name="rel_bias", shape=(2 * self.max_rel_pos + 1, self.h),
                                         initializer="zeros")
        # wq/wk/wv كلها تُبنى من نفس بُعد الإدخال (d_model)؛ wo يستقبل واقعياً
        # ctx مُعاد تشكيله لنفس d_model أيضاً — نفس input_shape يبنيها جميعاً.
        self.wq.build(input_shape)
        self.wk.build(input_shape)
        self.wv.build(input_shape)
        self.wo.build(input_shape)
        super().build(input_shape)

    def call(self, x, training=None):
        b, t = tf.shape(x)[0], tf.shape(x)[1]
        q = tf.reshape(self.wq(x), (b, t, self.h, self.dh))
        k = tf.reshape(self.wk(x), (b, t, self.hkv, self.dh))
        v = tf.reshape(self.wv(x), (b, t, self.hkv, self.dh))
        if self.hkv != self.h:
            rep = self.h // self.hkv
            k, v = tf.repeat(k, rep, axis=2), tf.repeat(v, rep, axis=2)
        q, k, v = [tf.transpose(a, [0, 2, 1, 3]) for a in (q, k, v)]

        scores = tf.matmul(q, k, transpose_b=True) * (float(self.dh) ** -0.5)
        pos = tf.range(t)
        rel = pos[:, None] - pos[None, :]
        idx = tf.clip_by_value(rel, -self.max_rel_pos, self.max_rel_pos) + self.max_rel_pos
        bias = tf.transpose(tf.gather(self.rel_bias, idx), [2, 0, 1])[None]
        scores = scores + tf.cast(bias, scores.dtype)

        if self.causal or self.window is not None:
            allowed = tf.ones_like(rel, dtype=tf.bool)
            if self.causal:
                allowed = tf.logical_and(allowed, rel >= 0)
            if self.window is not None:
                allowed = tf.logical_and(allowed, tf.abs(rel) <= self.window)
            neg = tf.cast(scores.dtype.min, scores.dtype)
            scores = tf.where(allowed[None, None], scores, neg)

        attn = self.drop(tf.nn.softmax(scores, axis=-1), training=training)
        ctx = tf.transpose(tf.matmul(attn, v), [0, 2, 1, 3])
        return self.wo(tf.reshape(ctx, (b, t, self.d_model)))

    def get_config(self):
        return {**super().get_config(), "d_model": self.d_model, "num_heads": self.h,
                "num_kv_heads": self.hkv, "max_rel_pos": self.max_rel_pos, "causal": self.causal,
                "window": self.window, "dropout": self.dropout_rate, "out_std": self.out_std}


@register
class SwiGLU(layers.Layer):
    """بلا dropout داخلي: الكتلة تُطبِّق مرّة واحدة فقط dropout على المسار المتبقّي."""

    def __init__(self, d_model, dff=None, out_std=0.02, **kw):
        super().__init__(**kw)
        self.d_model = d_model
        self.dff = dff or int(round(8 * d_model / 3 / 8) * 8)
        self.out_std = out_std
        self.w1 = layers.Dense(self.dff, use_bias=False)
        self.w3 = layers.Dense(self.dff, use_bias=False)
        self.w2 = layers.Dense(d_model, use_bias=False,
                                kernel_initializer=initializers.TruncatedNormal(stddev=out_std))

    def build(self, input_shape):
        self.w1.build(input_shape)
        self.w3.build(input_shape)
        self.w2.build(tuple(input_shape[:-1]) + (self.dff,))  # مدخل w2 هو ناتج w1*w3، ببُعد dff
        super().build(input_shape)

    def call(self, x):
        return self.w2(tf.nn.silu(self.w1(x)) * self.w3(x))

    def get_config(self):
        return {**super().get_config(), "d_model": self.d_model, "dff": self.dff, "out_std": self.out_std}


@register
class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, num_kv_heads=None, max_rel_pos=16, causal=False,
                 window=None, dropout=0.1, attn_dropout=0.0, num_layers_for_init=4, **kw):
        super().__init__(**kw)
        self.cfg = dict(d_model=d_model, num_heads=num_heads, num_kv_heads=num_kv_heads,
                         max_rel_pos=max_rel_pos, causal=causal, window=window, dropout=dropout,
                         attn_dropout=attn_dropout, num_layers_for_init=num_layers_for_init)
        out_std = 0.02 / np.sqrt(2.0 * num_layers_for_init)
        self.norm1, self.norm2 = RMSNorm(), RMSNorm()
        self.attn = RelativeGQAttention(d_model, num_heads, num_kv_heads, max_rel_pos,
                                         causal, window, attn_dropout, out_std)
        self.ffn = SwiGLU(d_model, out_std=out_std)
        self.drop1, self.drop2 = layers.Dropout(dropout), layers.Dropout(dropout)

    def build(self, input_shape):
        self.norm1.build(input_shape)
        self.attn.build(input_shape)
        self.norm2.build(input_shape)
        self.ffn.build(input_shape)
        super().build(input_shape)

    def call(self, x, training=None):
        x = x + self.drop1(self.attn(self.norm1(x), training=training), training=training)
        x = x + self.drop2(self.ffn(self.norm2(x)), training=training)
        return x

    def get_config(self):
        return {**super().get_config(), **self.cfg}

## 5) مساعدات القراءة (readout)

In [ ]:
@register
class LastToken(layers.Layer):
    def call(self, x):
        return x[:, -1, :]

    def compute_output_shape(self, s):
        return (s[0], s[2])


@register
class AttentionPool(layers.Layer):
    """تجميع بانتباه بمفتاح استعلام مُتعلَّم (يستخدم كل الرموز لا آخرها فقط)."""

    def build(self, input_shape):
        self.q = self.add_weight(name="query", shape=(int(input_shape[-1]),),
                                  initializer=initializers.RandomNormal(stddev=0.02))
        super().build(input_shape)

    def call(self, x):
        d = tf.cast(tf.shape(x)[-1], x.dtype)
        w = tf.nn.softmax(tf.einsum("btd,d->bt", x, self.q) / tf.sqrt(d), axis=-1)
        return tf.einsum("bt,btd->bd", w, x)

    def compute_output_shape(self, s):
        return (s[0], s[2])


## 6) رؤوس NIG (بلا انتباه طول-1 منحطّ، أوساط مُرتَّبة، ثقة صادقة)

هذه الطبقات الخام كما هي — القسم التالي (٧) هو ما تغيّر: تحويلها إلى
"نوع رأس" مُسجَّل بدل استدعاء مباشر مُثبَّت بالكود.


In [ ]:
@register
class NIGHead(layers.Layer):
    def __init__(self, hidden=128, dropout=0.1, nu_min=0.1, alpha_min=2.0, beta_min=0.01,
                 l2=1e-5, **kw):
        super().__init__(**kw)
        self.cfg = dict(hidden=hidden, dropout=dropout, nu_min=nu_min, alpha_min=alpha_min,
                         beta_min=beta_min, l2=l2)
        self.fc = layers.Dense(hidden, activation="swish", kernel_regularizer=regularizers.l2(l2))
        self.drop = layers.Dropout(dropout)
        self.out = layers.Dense(4)

    def build(self, input_shape):
        self.fc.build(input_shape)
        self.out.build(tuple(input_shape[:-1]) + (self.cfg["hidden"],))
        super().build(input_shape)

    def call(self, h, training=None):
        p = self.out(self.drop(self.fc(h), training=training))
        mu = p[:, 0:1]
        nu = tf.nn.softplus(p[:, 1:2]) + self.cfg["nu_min"]
        alpha = tf.nn.softplus(p[:, 2:3]) + self.cfg["alpha_min"]
        beta = tf.nn.softplus(p[:, 3:4]) + self.cfg["beta_min"]
        return mu, nu, alpha, beta

    def get_config(self):
        return {**super().get_config(), **self.cfg}


@register
class OrderedMeans(layers.Layer):
    """mu_high = mu_close + softplus(a);  mu_low = mu_close - softplus(b)."""

    def call(self, inputs):
        mu_c, raw_h, raw_l = inputs
        return mu_c + tf.nn.softplus(raw_h), mu_c, mu_c - tf.nn.softplus(raw_l)


@register
class NIGUncertainty(layers.Layer):
    """تعريفات Amini et al. (محفوظة للتوافق الخلفي)."""

    def call(self, inputs):
        nu, alpha, beta = inputs
        aleatoric = tf.sqrt(beta / (alpha - 1.0 + 1e-8))
        epistemic = tf.sqrt(beta / (nu * (alpha - 1.0) + 1e-8))
        return epistemic, aleatoric


@register
class ConfidenceHead(layers.Layer):
    """سلسة (بلا قصّ صلب -> بلا تدرّجات ميّتة)، مشروطة بمعاملات evidential،
    ومقطوعة التدرّج بالكامل (stop_gradient) لتضمن أنها لا تقود جذع التنبؤ أبداً."""

    def __init__(self, hidden=32, **kw):
        super().__init__(**kw)
        self.hidden = hidden
        self.d1 = layers.Dense(hidden, activation="relu")
        self.d2 = layers.Dense(1, activation="sigmoid")

    def build(self, input_shape):
        h_shape, nu_shape, alpha_shape, beta_shape = input_shape
        concat_dim = h_shape[-1] + nu_shape[-1] + alpha_shape[-1] + beta_shape[-1]
        self.d1.build(tuple(h_shape[:-1]) + (concat_dim,))
        self.d2.build(tuple(h_shape[:-1]) + (self.hidden,))
        super().build(input_shape)

    def call(self, inputs):
        h, nu, alpha, beta = inputs
        u = tf.concat([h, tf.math.log(nu), tf.math.log(alpha - 1.0), tf.math.log(beta)], axis=-1)
        return 0.05 + 0.93 * self.d2(self.d1(tf.stop_gradient(u)))

    def get_config(self):
        return {**super().get_config(), "hidden": self.hidden}


## 7) سجلّ الرؤوس (`HEAD_REGISTRY`) — نقطة التعديل الوحيدة لإضافة/تسمية مخرجات جديدة

كل نوع رأس هنا دالة واحدة: `(h, target, cfg) -> dict مخرجات`. لإضافة رأس
جديد لاحقاً (مثلاً تصنيف متعدد الفئات لنظام السوق): أضف دالة، سجّلها بـ
`@register_head_type("اسمك")`، ثم أضف اسمها إلى
`MODEL_CONFIG['head_types'][الهدف]` — بلا لمس أي شيء آخر في المعمارية.

استثناء واحد مقصود: `nig_regression` لا يُبنى من هذا السجلّ مباشرة لأنه
يحتاج تنسيقاً عابراً للأهداف (ترتيب high>=close>=low عبر `OrderedMeans`)
قبل أن تُحسَم قيمه النهائية — التنسيق نفسه في `build_nig_timenet_v2`
(القسم ٨)، لكن اسم الرأس ما زال يُقرأ من `head_types` مثل أي رأس آخر،
فحذفه من `head_types` يُعطِّله بلا تعديل كود.


In [ ]:
HEAD_REGISTRY = {}


def register_head_type(name):
    """مُزخرِف: يُسجِّل دالة بناء رأس تحت اسم يُستخدَم في MODEL_CONFIG['head_types']."""
    def deco(fn):
        HEAD_REGISTRY[name] = fn
        return fn
    return deco


def build_head_outputs(head_type, h, target, cfg):
    """يستدعي دالة الرأس المسجّلة تحت head_type. يرفع خطأً واضحاً إن كان
    الاسم غير مسجَّل — بدل فشل صامت لاحقاً بمخرج مفقود."""
    if head_type not in HEAD_REGISTRY:
        raise ValueError(
            f"نوع رأس غير مسجَّل: '{head_type}'. الأنواع المتاحة: {sorted(HEAD_REGISTRY)}. "
            f"سجّل دالة جديدة بـ @register_head_type('{head_type}') قبل استخدامه في head_types."
        )
    return HEAD_REGISTRY[head_type](h, target, cfg)


@register_head_type("binary_classification")
def build_binary_classification_head(h, target, cfg):
    """رأس تصنيف ثنائي (مثال: صعود/هبوط) — مطابق لاصطلاح خط الأنابيب
    head_name(target, 'class') = f'{target}_class'، فمفتاح المخرج هنا
    'y_{target}_class_logits' (احتمال بعد sigmoid، لا logit خام)."""
    hidden = cfg.get("class_head_hidden", 64)
    dropout = cfg.get("dropout", 0.1)
    x = layers.Dense(hidden, activation="relu", name=f"class_fc_{target}")(h)
    x = layers.Dropout(dropout, name=f"class_drop_{target}")(x)
    prob = layers.Dense(1, activation="sigmoid", name=f"y_{target}_class_logits", dtype="float32")(x)
    return {f"y_{target}_class_logits": prob}


@register_head_type("multiclass_classification")
def build_multiclass_classification_head(h, target, cfg):
    """رأس تصنيف متعدّد الفئات (مثال: نظام سوق هادئ/متقلّب/اتجاهي).
    عدد الفئات من cfg['n_classes'][target] أو cfg['n_classes'] الرقم المباشر."""
    n_classes_cfg = cfg.get("n_classes", 3)
    n_classes = n_classes_cfg[target] if isinstance(n_classes_cfg, dict) else n_classes_cfg
    hidden = cfg.get("class_head_hidden", 64)
    dropout = cfg.get("dropout", 0.1)
    x = layers.Dense(hidden, activation="relu", name=f"class_fc_{target}")(h)
    x = layers.Dropout(dropout, name=f"class_drop_{target}")(x)
    # softmax لا logits خام: classification_task_loss في trainer_framework يستدعي
    # sparse_categorical_crossentropy بـ from_logits=False (الافتراضي) — يتوقّع
    # توزيع احتمالات جاهزاً، تماماً كرأس التصنيف الثنائي (sigmoid لا logit خام).
    probs = layers.Dense(n_classes, activation="softmax", name=f"y_{target}_class_logits", dtype="float32")(x)
    return {f"y_{target}_class_logits": probs}

## 8) بناء النموذج

In [ ]:
# احتياطي build_nig_timenet_v2 الداخلي فقط، لاستدعاء مباشر بلا head_types ولا
# عبر MODEL_CONFIG (الذي يحدّد فعلياً تصنيفاً+انحداراً معاً — انظر MODEL_CONFIG أدناه).
DEFAULT_HEAD_TYPES = {"high": ["nig_regression"], "low": ["nig_regression"], "close": ["nig_regression"]}


def build_nig_timenet_v2(
    seq_len, n_features,
    d_model=128, num_layers=4, num_heads=4, num_kv_heads=2,
    patch_len=4, stride=2, kernel_sizes=(3, 5, 9, 17), max_rel_pos=16,
    dropout=0.1, attn_dropout=0.0, causal=False, window=None,
    use_instance_norm=True, use_decomposition=True, use_linear_path=True,
    price_targets=("high", "low", "close"), head_types=None,
    enforce_order=True, head_hidden=128,
    nu_min=0.1, alpha_min=2.0, beta_min=0.01,
    class_head_hidden=64, n_classes=3,
    norm_eps=1e-4, input_clip=None, stats_mode="full", input_noise_std=0.0, feature_dropout=0.0,
    linear_path_l2=1e-4, level_passthrough=False,
    name="nig_timenet_v2",
):
    """
    seq_len/n_features: شكل نافذة إطار زمني واحد (لا دمج متعدّد الأطر — انظر
    ملاحظة النطاق في دفتر main).

    head_types: {هدف: [أنواع رؤوس]} — عبر build_model_fn/MODEL_CONFIG (نقطة
    الدخول الفعلية، القسم ٩) يتضمّن تصنيفاً+انحداراً معاً افتراضياً. استدعاء
    هذه الدالة مباشرة بلا head_types يقع على DEFAULT_HEAD_TYPES (انحدار NIG
    فقط) بدلاً من ذلك. لإضافة/إزالة رأس لهدف: عدّل القاموس نفسه، مثلاً:
        head_types = dict(DEFAULT_HEAD_TYPES)
        head_types['close'] = ['nig_regression', 'binary_classification']

    خيارات مقاومة الحفظ (كلها افتراضياً = السلوك القديم؛ ANTI_MEMORIZATION_CONFIG في القسم ٩ يفعّلها معاً):
      norm_eps / input_clip / stats_mode: حرّاس InstanceNorm (راجع docstring الطبقة).
      feature_dropout: يُسقط **قناة ميزة كاملة** (كل خطواتها الزمنية) لكل عيّنة أثناء التدريب
                       (SpatialDropout1D) — يمنع الاعتماد على ميزة واحدة تصلح معرِّفاً للعيّنة.
      input_noise_std: ضجيج غاوسي على z المُطبَّع أثناء التدريب فقط — كل حقبة يرى النموذج نسخة مختلفة
                       قليلاً من نفس العيّنة فلا يستطيع حفظ قيمها الدقيقة.
      linear_path_l2: عقوبة L2 على المسار الخطّي (Flatten(z) → d_model: seq_len×n_features مُدخلاً).
      level_passthrough: InstanceNorm يمحو **مستوى** كل ميزة داخل النافذة (z = شكلها فقط)، فالفرق بين
                       عملة متقلّبة وأخرى هادئة (إشارة H003 نفسها) لا يصل للمحوّل إلا عبر مسار stats الجانبي.
                       True يضيف المدخل الخام بعد SymLog كقنوات إضافية لكل رمز وللمسار الخطّي — المحوّل
                       والمسار الخطّي يريان الشكل والمستوى معاً.
    الثلاثة الأولى تعمل على z قبل التفكيك والمسار الخطّي معاً؛ لا أثر لها في الاستدلال (training=False).
    """
    head_types = head_types or DEFAULT_HEAD_TYPES
    head_cfg = dict(
        dropout=dropout, class_head_hidden=class_head_hidden, n_classes=n_classes,
        head_hidden=head_hidden, nu_min=nu_min, alpha_min=alpha_min, beta_min=beta_min,
    )

    inputs = layers.Input(shape=(seq_len, n_features), name="input_sequence")

    if use_instance_norm:
        z, stats = InstanceNorm(eps=norm_eps, clip=input_clip, stats_mode=stats_mode, name="instance_norm")(inputs)
        if stats_mode == "none":
            stats = None
    else:
        z, stats = inputs, None
    if feature_dropout > 0:
        z = layers.SpatialDropout1D(feature_dropout, name="feature_dropout")(z)
    if input_noise_std > 0:
        z = layers.GaussianNoise(input_noise_std, name="input_noise")(z)

    if use_decomposition:
        trend, seasonal = CausalMultiScaleDecomp(kernel_sizes, name="decomp")(z)
        tok_in = layers.Concatenate(axis=-1, name="trend_seasonal")([trend, seasonal])
    else:
        tok_in = z
    lvl = None
    if level_passthrough:
        lvl = SymLog(name="level_symlog")(inputs)
        if feature_dropout > 0:
            lvl = layers.SpatialDropout1D(feature_dropout, name="level_feature_dropout")(lvl)
        if input_noise_std > 0:
            lvl = layers.GaussianNoise(input_noise_std, name="level_noise")(lvl)
        tok_in = layers.Concatenate(axis=-1, name="shape_and_level")([tok_in, lvl])

    x = PatchEmbedding(d_model, patch_len, stride, name="patch_embed")(tok_in)
    for i in range(num_layers):
        x = TransformerBlock(d_model, num_heads, num_kv_heads, max_rel_pos, causal, window,
                              dropout, attn_dropout, num_layers_for_init=num_layers,
                              name=f"block_{i + 1}")(x)
    x = RMSNorm(name="final_norm")(x)

    parts = [LastToken(name="last_token")(x), AttentionPool(name="attn_pool")(x)]
    if stats is not None:
        parts.append(layers.Dense(d_model, activation="gelu", name="stats_proj")(stats))
    h = layers.Concatenate(name="readout_concat")(parts)
    h = layers.Dense(d_model, activation="gelu", name="readout_fc")(h)
    if use_linear_path:
        lin_in = z if lvl is None else layers.Concatenate(axis=-1, name="linear_shape_and_level")([z, lvl])
        lin = layers.Dense(d_model, kernel_regularizer=regularizers.l2(linear_path_l2), name="linear_path")(
            layers.Flatten(name="flatten_window")(lin_in))
        h = layers.Add(name="add_linear_path")([h, lin])
    h = RMSNorm(name="trunk_norm")(h)
    h = layers.Dropout(dropout, name="trunk_drop")(h)

    # -- رؤوس nig_regression: تُبنى أولاً بمعزل عن التسمية النهائية، لأن
    #    enforce_order يحتاج القيم الخام الثلاث معاً قبل الحسم --
    nig_targets = [t for t in price_targets if "nig_regression" in head_types.get(t, [])]
    raw_nig = {}
    for t in nig_targets:
        nig = NIGHead(head_hidden, dropout, nu_min, alpha_min, beta_min, name=f"nig_{t}")
        raw_nig[t] = nig(h)  # (mu_raw, nu, alpha, beta)

    apply_order = enforce_order and {"high", "low", "close"}.issubset(nig_targets)
    if apply_order:
        mu_h, mu_c, mu_l = OrderedMeans(name="ordered_means")(
            [raw_nig["close"][0], raw_nig["high"][0], raw_nig["low"][0]])
        mus = {"high": mu_h, "close": mu_c, "low": mu_l}
    else:
        mus = {t: raw_nig[t][0] for t in nig_targets}

    outputs = {}
    for t in nig_targets:
        _, nu, alpha, beta = raw_nig[t]
        epi, ale = NIGUncertainty(name=f"unc_{t}")([nu, alpha, beta])
        conf = ConfidenceHead(name=f"conf_{t}")([h, nu, alpha, beta])
        outputs.update({
            f"y_{t}": mus[t], f"y_{t}_nu": nu, f"y_{t}_alpha": alpha, f"y_{t}_beta": beta,
            f"y_{t}_epistemic": epi, f"y_{t}_aleatoric": ale, f"y_{t}_confidence": conf,
        })

    # -- كل أنواع الرؤوس الأخرى (تصنيف، ...): مستقلّة لكل هدف، عبر السجلّ --
    for t in price_targets:
        for head_type in head_types.get(t, []):
            if head_type == "nig_regression":
                continue
            outputs.update(build_head_outputs(head_type, h, t, head_cfg))

    return Model(inputs, outputs, name=name)


## 9) `MODEL_CONFIG` + `build_model_fn` — نقطة الدخول من دفتر `main`

على عكس النسخة السابقة (حيث كانت `MODEL_CONFIG` بمفاتيح معمارية قديمة غير
مستخدَمة فعلياً، و`seq_len`/`n_features` مُثبَّتتين بالكود)، هنا:

* كل مفاتيح `MODEL_CONFIG` تُطابق أسماء معاملات `build_nig_timenet_v2` فعلياً.
* `seq_len`/`n_features` وسيطان **إلزاميان** لـ`build_model_fn` — يُشتقّان من
  بيانات خط الأنابيب الفعلية في دفتر main (`dataset['window_sizes'][model_tf]`
  و`len(dataset['feature_order'])`)، لا يُخمَّنان هنا.
* `ANTI_MEMORIZATION_CONFIG` (PR #7): إعداد أصغر ومُنظَّم يُدمَج فوق `MODEL_CONFIG` —
  `build_model_fn(seq_len, n_features, config=ANTI_MEMORIZATION_CONFIG)`، أو `ANTI_MEMORIZATION = True` في main.


In [ ]:
MODEL_CONFIG = dict(
    d_model=128,
    num_layers=4,
    num_heads=4,
    num_kv_heads=2,
    patch_len=4,
    stride=2,
    kernel_sizes=(3, 5, 9, 17),
    max_rel_pos=16,
    dropout=0.1,
    attn_dropout=0.0,
    causal=False,
    window=None,
    use_instance_norm=True,
    use_decomposition=True,
    use_linear_path=True,
    price_targets=("high", "low", "close"),
    # رأسا تصنيف وانحدار معاً لكل هدف — يطابق enabled_heads الافتراضي في
    # خط الأنابيب. عطّل التصنيف بإعادة هذا لـ DEFAULT_HEAD_TYPES (نسخة).
    head_types={
        "high": ["nig_regression", "binary_classification"],
        "low": ["nig_regression", "binary_classification"],
        "close": ["nig_regression", "binary_classification"],
    },
    enforce_order=True,
    head_hidden=128,
    nu_min=0.1,
    alpha_min=2.0,
    beta_min=0.01,
    class_head_hidden=64,
    n_classes=3,
    # مقاومة الحفظ — القيم هنا = السلوك القديم تماماً (نموذج مُدرَّب سابقاً يُستأنف بلا تغيير في المعمارية).
    norm_eps=1e-4,
    input_clip=None,
    stats_mode="full",
    input_noise_std=0.0,
    feature_dropout=0.0,
    linear_path_l2=1e-4,
    level_passthrough=False,
)

# ── إعداد مقاومة الحفظ (PR #7) — يُدمَج فوق MODEL_CONFIG عبر build_model_fn(..., config=ANTI_MEMORIZATION_CONFIG)
# (أو ANTI_MEMORIZATION=True في دفتر main). كل قيمة هنا مُختبَرة فعلياً على بيانات عملات حقيقية يومية
# (47 عملة، 2018→2026)، مع سيناريوهات «بيانات قليلة» و«تطبيع سيئ» و«تسميات مخلوطة» — الأرقام والتجارب
# التي رُفضت في docs/research/anti_memorization_pr7.md.
ANTI_MEMORIZATION_CONFIG = dict(
    d_model=64,
    num_layers=2,
    head_hidden=64,
    class_head_hidden=32,
    dropout=0.25,
    input_clip=4.0,
    stats_mode="symlog",
    input_noise_std=0.1,
    feature_dropout=0.1,
    linear_path_l2=1e-3,
)


def build_model_fn(seq_len, n_features, config=None):
    """نقطة الدخول العامة لبناء النموذج. `config` يُدمَج فوق MODEL_CONFIG
    (المفاتيح غير المذكورة تبقى على قيمتها الافتراضية)."""
    cfg = dict(MODEL_CONFIG)
    if config:
        cfg.update(config)
    return build_nig_timenet_v2(seq_len=seq_len, n_features=n_features, **cfg)

## 10) اختبار ذاتي (بلا بيانات حقيقية) — يثبت أن السجلّ والتوسعة والحفظ الكامل تعمل

يبني نموذجاً بالإعداد الافتراضي (تصنيف+انحدار معاً، تحقّق من مفاتيح
المخرجات واحتمالات صالحة)، يعطّل التصنيف بالكامل عبر `config` ليثبت أن
الاتجاه المعاكس يعمل أيضاً، يوسّع بإضافة رأس متعدّد الفئات لهدف واحد،
ثم — الأهم — **يحفظ نموذجاً مبنياً كاملاً إلى القرص ويُعيد تحميله بدون أي
استدعاء لاحق لـ`build_model_fn`**، للتأكّد أن `model.save(...)` يكفي وحده
(لا حاجة لإعادة البناء ثم `load_weights` فقط).

In [ ]:
def run_model_selftests(verbose: bool = True) -> bool:
    seq_len, n_features = 32, 38
    batch = 4
    x = np.random.randn(batch, seq_len, n_features).astype("float32")

    # 1) الإعداد الافتراضي: تصنيف + انحدار معاً لكل هدف (يطابق enabled_heads
    #    الافتراضي في خط الأنابيب) — 21 مخرج NIG + 3 احتمالات تصنيف ثنائي
    model_default = build_model_fn(seq_len, n_features)
    out = model_default(x, training=False)
    expected_default = set()
    for t in ("high", "low", "close"):
        expected_default |= {f"y_{t}", f"y_{t}_nu", f"y_{t}_alpha", f"y_{t}_beta",
                              f"y_{t}_epistemic", f"y_{t}_aleatoric", f"y_{t}_confidence",
                              f"y_{t}_class_logits"}
    assert set(out.keys()) == expected_default, (set(out.keys()), expected_default)
    for k, v in out.items():
        assert v.shape == (batch, 1), f"{k}: {v.shape}"
    for t in ("high", "low", "close"):
        p = out[f"y_{t}_class_logits"].numpy()
        assert np.all((p >= 0.0) & (p <= 1.0)), f"y_{t}_class_logits ليست احتمالات صالحة: {p}"

    # 2) high>=close>=low يجب أن يصمد فعلياً (enforce_order) — بمعزل عن التصنيف
    h, c, l = out["y_high"].numpy(), out["y_close"].numpy(), out["y_low"].numpy()
    assert np.all(h >= c - 1e-5) and np.all(c >= l - 1e-5), "OrderedMeans لم يحفظ الترتيب"

    # 3) تعطيل التصنيف بالكامل عبر config — يعيد سلوك النسخة السابقة تماماً
    reg_only = {t: ["nig_regression"] for t in ("high", "low", "close")}
    model_reg_only = build_model_fn(seq_len, n_features, config={"head_types": reg_only})
    out_reg_only = model_reg_only(x, training=False)
    assert not any(k.endswith("_class_logits") for k in out_reg_only), "تعطيل التصنيف لم يُطبَّق"
    assert len(out_reg_only) == 21

    # 4) توسعة إضافية بلا لمس كود المعمارية: رأس متعدّد فئات فوق ما هو مُفعَّل أصلاً
    extended_heads = {"high": ["nig_regression", "binary_classification"],
                       "low": ["nig_regression", "binary_classification"],
                       "close": ["nig_regression", "binary_classification", "multiclass_classification"]}
    model_ext = build_model_fn(seq_len, n_features,
                                config={"head_types": extended_heads, "n_classes": 3})
    out_ext = model_ext(x, training=False)
    assert out_ext["y_close_class_logits"].shape == (batch, 3), "الرأس متعدّد الفئات لم يظهر بالشكل الصحيح"
    probs = out_ext["y_close_class_logits"].numpy()
    assert np.allclose(probs.sum(axis=-1), 1.0, atol=1e-4), "مخرج multiclass ليس توزيع احتمالات (softmax)"

    # 5) نوع رأس غير مسجَّل يرفع خطأً واضحاً لا فشلاً صامتاً
    try:
        build_model_fn(seq_len, n_features, config={"head_types": {"close": ["does_not_exist"]}})
        raise AssertionError("كان يجب رفع ValueError لنوع رأس غير مسجَّل")
    except ValueError:
        pass

    # 6) حفظ كامل + تحميل بلا أي إعادة بناء (لا build_model_fn ولا custom_objects) —
    #    كل الطبقات المخصَّصة مُسجَّلة (register_keras_serializable) وتُبنى صراحةً
    #    في build() (لا انتظار أول call)، فالحفظ الأصلي (native .keras) يكفي وحده.
    import os
    import tempfile
    with tempfile.TemporaryDirectory() as d:
        path = os.path.join(d, "model.keras")
        model_default.save(path)
        reloaded = tf.keras.models.load_model(path)
        out_reloaded = reloaded(x, training=False)
        assert set(out_reloaded.keys()) == expected_default
        for k in expected_default:
            assert np.allclose(out[k].numpy(), out_reloaded[k].numpy(), atol=1e-5), (
                f"تباين بعد التحميل في {k} — النموذج المُعاد لا يطابق الأصلي")

    # 7) إعداد مقاومة الحفظ: يُبنى ويُحفظ ويُحمَّل كاملاً، عشوائي في التدريب فقط وحتمي في الاستدلال
    model_am = build_model_fn(seq_len, n_features, config=ANTI_MEMORIZATION_CONFIG)
    out_a = model_am(x, training=False)
    out_b = model_am(x, training=False)
    for k in out_a:
        assert np.allclose(out_a[k].numpy(), out_b[k].numpy()), f"{k}: الاستدلال ليس حتمياً"
    tr_a = model_am(x, training=True)["y_close"].numpy()
    tr_b = model_am(x, training=True)["y_close"].numpy()
    assert not np.allclose(tr_a, tr_b), "الضجيج/الإسقاط لا يعملان أثناء التدريب"
    with tempfile.TemporaryDirectory() as d:
        path = os.path.join(d, "model_am.keras")
        model_am.save(path)
        out_re = tf.keras.models.load_model(path)(x, training=False)
        for k in out_a:
            assert np.allclose(out_a[k].numpy(), out_re[k].numpy(), atol=1e-5), f"{k}: تباين بعد التحميل"

    # 8) stats_mode="symlog" يحصر مستوى ميزة غير مُطبَّعة (سعر خام ~10^5) في بضع وحدات بدل تمريره كما هو
    x_lvl = x.copy()
    x_lvl[..., 0] = 6.5e4 + 100.0 * x_lvl[..., 0]
    _, st_full = InstanceNorm(stats_mode="full")(tf.constant(x_lvl))
    _, st_sym = InstanceNorm(stats_mode="symlog")(tf.constant(x_lvl))
    assert float(tf.reduce_max(tf.abs(st_full))) > 6e4 and float(tf.reduce_max(tf.abs(st_sym))) < 12.0

    if verbose:
        print(f"✅ نجحت كل اختبارات النموذج الذاتية ({len(expected_default)} مخرجاً افتراضياً "
              f"شاملة التصنيف، تعطيل/توسعة عبر config فقط، وحفظ+تحميل كامل بلا إعادة بناء، وإعداد مقاومة الحفظ)")
    return True

run_model_selftests()